In [ ]:
import ee
import os
import geemap
from dotenv import load_dotenv
load_dotenv()

# Initialize the Earth Engine module.
PROJECT_ID = os.getenv("PROJECT_ID")
ee.Initialize(project = os.getenv("PROJECT_ID"))

# Define Denmark Area
DENMARK = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Denmark')).geometry()


In [ ]:
# Calculate area of an image in km²
def dw_area(img):
    area_image = img.multiply(ee.Image.pixelArea())
    area = area_image.reduceRegion( 
        reducer = ee.Reducer.sum(),
        scale = 10,
        maxPixels = 1e13,
    )
    return ee.Number(area.get('label')).divide(pow(10,6))

# Calculate change matrix
# 9x9 matrix area classified per class vs area classified per class other year
def change_matrix(img, img_prev):
    for i in range(9):
        for j in range(9):
            dw_transition = img.eq(i).And(img_prev.eq(j))
            dw_transition_area = dw_area(dw_transition)
            display(dw_transition_area)


In [ ]:
YEAR = 25

# DANISH AGRICULTURE AGENCY
dataset = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR))
dataset_prev = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR - 1))
# Transform data to raster format for lighter operations
daa_img = ee.Image.constant(0).paint(dataset, 1)
daa_img_prev = ee.Image.constant(0).paint(dataset_prev, 1)
# Image with are that changed from DAA (1 - new field, -1 - removed field)
daa_change_img = daa_img.subtract(daa_img_prev)
daa_change_img = daa_change_img.updateMask(daa_change_img.neq(0))


# DYNAMIC WORLD CLASSIFICATION OF DAA AREA
dw_img = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR) + '_mode').clipToCollection(dataset)
dw_img_prev = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR - 1) + '_mode').clipToCollection(dataset_prev)

## Change Matrix

### 2024 - 2025

|      | Water | Trees   | Grass   | Flooded Vegetation | Crops    | Shrub & Scrub | Built  | Bare  | Snow & Ice |
| ---  | ---   | ---     | ---     | ---                | ---      | ---           | ---    | ---   | ---        |
| 2024 | 59.55 | 2875.20 | 2748.15 | 49.60              | 21001.98 |  59.27        | 240.01 | 12.11 | 1.25       |
| 2025 | 64.06 | 2440.08 | 5470.97 | 27.95              | 18589.29 | 154.01        | 222.01 | 22.65 | 0.89       |

#### Not Changed Area

What are the common class changes? Which classifications turn into which?

| 2025/2024          | Water | Trees   | Grass    | Flooded Vegetation | Crops    | Shrub & Scrub | Built  | Bare | Snow & Ice |
| ---                | ---   | ---     | ---      | ---                | ---      | ---           | ---    | ---  | ---        |
| Water              | 49.09 | 4.74    | 2.35     |       5.09         | 1.66     |      0.03     | 0.06   | 0.09 |    0.02    |
| Trees              | 1.74  | 1936.57 | 97.08    |       7.70         | 354.85   |     4.36      | 8.14   | 0.04 |    0.03    |
| Grass              | 3.77  | 606.14  | 2191.07  |      13.78         | 2603.96  |     10.14     | 18.27  | 0.32 |    0.11    |
| Flooded Vegetation | 0.71  | 3.95    | 1.75     |      20.75         |  0.19    |     0.12      |  0.01  | 0.03 |    0.00    |
| Crops              | 2.06  | 195.13  | 419.13   |     0.61           | 17903.91 |    5.91       | 30.98  | 0.65 |    0.07    |
| Shrub & Scrub      | 0.16  | 70.23   | 16.50    |     1.09           | 28.02    |    33.23      | 0.70   | 0.35 |    0.42    |
| Built              | 0.03  | 5.74    | 3.00     |     0.01           | 44.42    |    0.20       | 164.35 | 0.30 |    0.04    |
| Bare               | 0.08  | 0.78    | 1.16     |     0.17           |  6.97    |    3.84       |  0.32  | 8.80 |    0.23    |
| Snow & Ice         | 0.01  | 0.03    | 0.02     |     0.01           |  0.15    |    0.21       |  0.07  | 0.07 |    0.31    |

In [16]:
change_matrix(dw_img, dw_img_prev)

#### New Fields Area

From the new field area, how did classification change? How was it classified before? How is it classified now?

New Field Area from 2024-2025: 90.54 km²

In [ ]:
change_matrix(dw_img.And(daa_change_img.eq(1)), dw_img_prev.And(daa_change_img.eq(1)))

#### Removed Fields Area

From the removed field area, how did classification change? How was it classified before? How is it classified now?

Removed Field Area from 2024-2025: 148.18 km²

In [ ]:
change_matrix(dw_img.And(daa_change_img.eq(-1)), dw_img_prev.And(daa_change_img.eq(-1)))